### Why this exists

litesearch grew four ways to answer the same question. A reader who wants *"search my PDFs"* has
to pick a chunk granularity, an encoder, a dtype, a retrieval strategy, whether to build a tree,
and whether to build an entity graph — six decisions before the first result, five of which
`evals/` already settled.

`Index` makes those five and leaves one:

| decision | what `Index` does | measured basis (weighted section MRR, 3 genres x 120 queries) |
|---|---|---|
| encoder | static `potion-retrieval-32M` | spread across four encoders is 0.018–0.046, and the static one *wins* astrology |
| dtype | `float16`, matched to the store | a mismatch returns rowid order **silently** — the one trap that fails quietly |
| granularity | 512 characters | +0.06 to +0.12 over page-sized chunks |
| FTS leg | `pre()` — keywords, wildcards, OR | +0.016 to +0.093 |
| vector leg | HNSW ANN | −0.005 quality, large speedup |
| tree | always built | ranking is a wash (−0.052 to +0.011), so there is no tradeoff to weigh — `toc`/`read`/`sections` come free |
| **rerank** | **yours** (`rerank=True`) | **+0.026 to +0.077**, positive in all twelve paired cells — the one lever worth a decision |

Nothing underneath moved. `Index.db` is the `database()` you would have built by hand and every
method on it still works. This is a front door, not a wall.


In [ ]:
#| default_exp api

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
from fastcore.all import Path, patch
from litesearch.core import database, rerank_hits
from litesearch.data import dir2chunks, pkg2chunks
from litesearch.tree import DOC_EXTS
from litesearch.utils import static_embedder, doc_encoder, query_encoder

## `Index`

Three lines to a searchable corpus:

```python
ix = Index('kb.db')
ix.add('docs/')
ix.search('how does batching work')
```

`Index` owns an encoder, a chunk store and a document tree. It is deliberately not clever: every
method is a thin call into `litesearch.core` or `litesearch.tree` with the arguments the
evaluation argues for already filled in.


In [ ]:
#| export
DTYPE, HIT_COLS, RERANK_FANOUT = np.float16, ['content', 'metadata', 'node_id', 'page', 'heading', 'doc_id'], 30

In [ ]:
#| export
class Index:
    '''A corpus you can search: ingest files, code or text, then query with a string.
    The defaults are not a matter of taste — they are the configuration `evals/` measures as best
    or tied-best across three genres. Everything is overridable, and `self.db` is the plain
    `database()` underneath.
    '''
    def __init__(self,
                 path=':memory:',     # sqlite file; omit for in-memory
                 encoder=None,        # anything `doc_encoder` takes (default: static potion-retrieval-32M)
                 name:str='store',    # chunk table, if you want several corpora in one file
                 ann:bool=True,       # maintain an HNSW index for the vector leg
                 ):
        self.path, self.name, self.ann = str(path), name, ann
        self.db = database(path)
        self.encoder = static_embedder() if encoder is None else encoder
        self._doc, self._qry = doc_encoder(self.encoder), query_encoder(self.encoder)
        self.t = self.db.get_tree(name, ann=ann)
        self.store = self.t.store

    def emb(self, texts) -> np.ndarray:
        'Document vectors, cast to the dtype the store actually holds.'
        return np.asarray(self._doc(list(texts)), dtype=DTYPE)

    def qemb(self, q:str) -> bytes:
        'One query vector, as the bytes the low-level `search` wants.'
        return np.asarray(self._qry([q]), dtype=DTYPE)[0].tobytes()

    def __len__(self): return self.db.q(f'select count(*) as n from [{self.name}]')[0]['n']

    @property
    def docs(self) -> list:
        'One row per ingested document — title, source, kind, pages.'
        return list(self.t.docs())

    def __repr__(self):
        return f'Index(path={self.path!r}, chunks={len(self):,}, docs={len(self.docs)})'

### Ingest

`add` takes whatever you have: a directory, a single file, a string, a list of strings, or a
`{title: text}` mapping. Documents go through `litesearch.tree`, so each keeps its heading
structure and every chunk remembers the section it came from.

Source code goes through `add_code` instead. Its tree is module › class › function and comes from
the AST rather than from headings — a different parser, so a different door.


In [ ]:
#| export
def _texts(src) -> list: return [src] if isinstance(src, str) else list(src)

def _title(t:str, i:int, n:int=60) -> str:
    'First non-empty line, trimmed — so `toc()` over raw strings still reads like a listing.'
    ln = next((l.strip() for l in (t or '').splitlines() if l.strip()), '')
    return ln[:n] or f'text-{i+1}'

@patch
def add(self:Index,
        src,                    # directory, file path, string, list of strings, or {title: text}
        types:str=DOC_EXTS,     # extensions to pick up when `src` is a directory
        **kw                    # forwarded to add_dir / add_file / add_doc
        ) -> int:
    'Ingest documents or text. Returns the number of chunks the store gained.'
    before = len(self)
    if isinstance(src, (str, Path)) and Path(src).exists():
        p = Path(src)
        if p.is_dir(): self.db.add_dir(str(p), types=types, store=self.name, emb_fn=self.emb, **kw)
        else:          self.db.add_file(str(p), store=self.name, emb_fn=self.emb, **kw)
    else:
        items = src if isinstance(src, dict) else {_title(t, i): t for i, t in enumerate(_texts(src))}
        for title, txt in items.items():
            self.db.add_doc(txt, title=title, store=self.name, emb_fn=self.emb, **kw)
    return len(self) - before

In [ ]:
#| export
@patch
def add_code(self:Index,
             src,       # a source directory, or an installed package name
             **kw       # forwarded to dir2chunks / pkg2chunks
             ) -> int:
    'Ingest source code through the AST path — top-level functions, classes and assignments.'
    chunks = dir2chunks(str(src), **kw) if Path(src).exists() else pkg2chunks(str(src), **kw)
    rows = [dict(content=c['content'], metadata=str(c.get('metadata') or {}))
            for c in chunks if (c.get('content') or '').strip()]
    if not rows: return 0
    for r, v in zip(rows, self.emb([r['content'] for r in rows])):
        r['embedding'] = np.asarray(v, dtype=DTYPE).tobytes()
    self.store.insert_all(rows, upsert=True, hash_id='id', hash_id_columns=['content'])
    if self.ann: self.store.rebuild_index()
    return len(rows)

### Search

`search` is the hybrid the whole library is for: FTS5 keyword and SIMD vector, merged with
Reciprocal Rank Fusion. Pass a string, get chunks back.

`rerank=True` is the one knob the evaluation asks you to think about. It fetches `RERANK_FANOUT`
candidates and reorders them with a flashrank cross-encoder — worth +0.026 to +0.077 weighted MRR,
positive in all twelve paired cells measured, at roughly 10x the latency and a small model
download on first use. The fanout is load-bearing: reranking ten candidates only reorders ten, and
the measured gain comes from reranking thirty.


In [ ]:
#| export
@patch
def search(self:Index,
           q:str,                # query string
           limit:int=10,         # hits to return
           rerank:bool=False,    # reorder with a flashrank cross-encoder (see above)
           **kw                  # forwarded to Database.search
           ) -> list:
    'Hybrid keyword + vector search over the chunk store.'
    if not (q or '').strip(): return []
    qv = self.qemb(q)
    cols = list(dict.fromkeys(HIT_COLS + (kw.pop('columns', None) or [])))
    n = max(limit, RERANK_FANOUT) if rerank else limit
    hits = self.db.search(q, qv, columns=cols, table_name=self.name, limit=n,
                          dtype=DTYPE, ann=self.ann, **kw) or []
    return rerank_hits(q, hits, None, limit) if rerank else hits

### Read

The tree is built at ingest whether or not you ask for it, because ranking-wise it is free. These
four methods are what it buys, and the reason to keep it even though it does not move MRR.

| method | what it answers |
|---|---|
| `toc()` | "what is even in this corpus?" — touches no embeddings |
| `sections(q)` | "which *sections* are about this", not which 512 characters |
| `read(node_id)` | "give me that whole section", reassembled from its chunks |
| `context(q)` | operative sections plus what they connect to, composed for a model |

On the Sanskrit corpus `context` roughly *doubles* verse-level recall over plain chunk search
(0.190 → 0.340) — the largest single effect measured anywhere in `evals/`, and larger than any
encoder swap. Section assembly is doing real work even where section *ranking* is not.


In [ ]:
#| export
@patch
def toc(self:Index, doc:str=None, **kw) -> list:
    'The corpus as a nested tree of titles and page ranges. Touches no embeddings.'
    return self.db.toc(doc, store=self.name, **kw)

@patch
def sections(self:Index, q:str, limit:int=5, **kw) -> list:
    'Ranked sections rather than chunks, each with snippets and a `read` handle.'
    return self.db.sections(q, self.qemb(q), limit=limit, store=self.name, **kw)

@patch
def read(self:Index, node_id:str, **kw) -> dict:
    'One whole section, reassembled — the unit to hand a model instead of fragments.'
    return self.db.read(node_id, store=self.name, **kw)

@patch
def context(self:Index, q:str, **kw) -> dict:
    'Operative sections plus what they connect to, composed for a prompt.'
    return self.db.context(q, self.qemb(q), store=self.name, **kw)

## Worked example

Five short documents, ingested as raw text so the cell runs anywhere.


In [ ]:
ix = Index()
ix.add({'Batching': 'Requests are batched by the scheduler before they reach the model. '
                    'Batch size trades latency for throughput.',
        'Caching':  'A prompt cache stores the prefix of a request so that repeated prefixes '
                    'skip recomputation entirely.',
        'Sharding': 'Large models are sharded across devices; each shard holds a slice of every '
                    'weight matrix.',
        'Metrics':  'Throughput is tokens per second. Latency is time to first token.',
        'Retries':  'Failed requests are retried with exponential backoff and a jitter term.'})
ix

In [ ]:
[(h['doc_id'], h['content'][:44]) for h in ix.search('batch size and throughput', limit=3)]

The same query rolled up to sections rather than chunks:

In [ ]:
[(s['node_id'], (s['snippets'] or [''])[0][:44])
 for s in ix.sections('batch size and throughput', limit=2)]

In [ ]:
[d['title'] for d in ix.docs]

## The layer underneath

`Index` is one of two routes and the other is not going away. Reach past it when you want
something it deliberately decided for you:

| you want | use |
|---|---|
| SQL over your own columns, joins, filters | `ix.db` — a plain `database()` |
| a different encoder, or float32 vectors | `database()` + `get_store()` directly |
| several stores, custom FTS tokenizers | `db.get_store(name=..., tokenize=...)` |
| entity-graph retrieval for bridge queries | `db.graph_search` — see [graph](05_graph.ipynb) |
| Sanskrit script folding, metre, verse trees | `litesearch.sanskrit` — see [sanskrit](09_sanskrit.ipynb) |

The graph leg is the one capability `Index` does not expose at all, and that is measured rather
than an oversight. On ordinary known-item queries it costs 0.070 to 0.160 weighted MRR — negative
in every cell, every genre and every flavour, monotonically worse as its weight rises. On the
bridge query set built specifically to favour it, where the answer shares no token with the
question, it buys roughly +0.04 target MRR and +0.12 hit@1 — on one genre of three, while losing
0.10 to 0.16 on the ordinary questions and running 3–4x slower. Call `db.graph_search` by name
when you know your traffic looks like that. Do not reach for it by default.


## Downstream: vishalakshi is the first test

[vishalakshi](https://github.com/vedicreader/vishalakshi) builds a litesearch-backed vault, and it
is the nominated candidate for porting onto `Index` — a real caller is a better test of "did this
actually remove decisions" than any example in this notebook. What the port should tell us:

- **Does `add` cover the ingest it does by hand?** The Sanskrit `Profile`s register at import, so
  `add_file` already picks the reader, `verse` tree mode, `VerseChunker` and the metrical facets
  without arguments. `Index.add` forwards to it unchanged, which means the vault should need no
  chunker and no tree wiring of its own. If it does, that is a gap in `Index`, not in the caller.
- **Is the default encoder wrong for it?** `Index` defaults to `potion-multilingual-128M`, to enable vishalakshi on the Gītā full-stack table
- **Does `context()` carry the vault's read path?** It is the method with the largest measured
  effect on that corpus (0.190 → 0.340 verse recall against plain chunk search), so a vault that
  assembles passages for a reader should be built on `context`/`read`, not on `search`.

Note this was written without reading vishalakshi's source, so treat the three points as the
questions to answer during the port rather than as findings about its current code.

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()